# Saarthi Finance ML Pipeline
This notebook contains the complete feature engineering, training, cross-validation, and serialization pipeline for Saarthi's three core operational predictive models:

1. **BD Velocity:** A HistGradientBoostingRegressor optimizing MAE loss to forecast the median days required to close active deals.
2. **Leakage Propensity:** A RandomForestClassifier with class balance weights predicting deals likely to close without generating bills.
3. **Payment Collection Risk:** A RandomForestClassifier predicting the likelihood of invoice collection delays.

In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
import sklearn.metrics as metrics
import joblib
import sys

load_dotenv()

True

## 1. Load and Verify Ingestion Dataset

In [2]:
csv_path = 'miss/training_dataset_clean.csv'
if not os.path.exists(csv_path):
    csv_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'miss', 'training_dataset_clean.csv')

df = pd.read_csv(csv_path)
print(f"Successfully loaded {len(df)} records for training.")
df.head()

Successfully loaded 13874 records for training.


,id,companyName,bdMemberName,teamLeaderName,franchiseeName,positionName,placementFees,enquiryStatus,dateOfAllocation,invoice_billNumber,invoice_billDate,invoice_grossRevenue,invoice_netRevenue,invoice_amountReceived,invoice_financialYear
0,120006,APEX INFOTECH INDIA PVT. LTD.,Rajalaxmi Das Das,Avadai Esakki Muthu Sundaram Marthuvar,Yashvi Pragneshkumar Shah,HR Executive,8.33,closed,2024-06-24,270389/G/24-25,2024-06-11,17992.8,7871.8,21232.0,2024-2025
1,120008,JAYATMA TECHNOLOGIES,Komal Suresh Bhanushali,Avadai Esakki Muthu Sundaram Marthuvar,Sandeep,Accountant,8.33,closed,2024-05-31,4563263,2025-06-07,8000.0,8000.0,70000.0,2025-2026
2,120009,ZENNERA CLINICS,Mama . Paltasingh,Pune . Office,Razia Begum,Accountant,7.00,closed,2024-10-12,NaN,NaN,NaN,NaN,NaN,NaN
3,120010,XYZ SERVICES,Shreya Santosh Talashilkar,Surbhi Vinod Jain,Corporate Comrade Consultancy,Accountant,16.00,closed,2024-05-31,NaN,NaN,NaN,NaN,NaN,NaN
4,120011,TEMA BUSINESS SYSTEMS PVT. LTD.,Komal Suresh Bhanushali,Surbhi Vinod Jain,Unknown,HR Executive,8.33,closed,2024-05-09,270189/G/24-25,2024-05-09,2499.0,1093.0,2189.0,2024-2025


## 2. Base Feature Engineering & Cleaning
Convert date timestamps, bucket fee bands, map industry classifications, and fit base label encoders.

In [3]:
df['dateOfAllocation'] = pd.to_datetime(df['dateOfAllocation'])
df['bill_date'] = pd.to_datetime(df['invoice_billDate'])
df['days_to_close'] = (df['bill_date'] - df['dateOfAllocation']).dt.days

def get_industry(position):
    pos = str(position).lower()
    if any(w in pos for w in ['developer', 'software', 'tech', 'engineer', 'it', 'java', 'python', 'php', 'analyst']):
        return 'IT & Software'
    if any(w in pos for w in ['sales', 'marketing', 'bd', 'business development', 'retail', 'account manager']):
        return 'Sales & Marketing'
    if any(w in pos for w in ['finance', 'accountant', 'accounts', 'audit', 'tax', 'banking']):
        return 'Finance & Accounts'
    if any(w in pos for w in ['hr', 'recruiter', 'admin', 'human resources', 'operations']):
        return 'HR & Operations'
    return 'Other Services'

# Prefer the database human-assigned industry column when available, fall back to keyword matching
keyword_industry = df['positionName'].apply(get_industry)
if 'industry' in df.columns:
    df['industry'] = df['industry'].fillna(keyword_industry)
    # Group rare industry terms into "Other"
    counts = df['industry'].value_counts()
    rare = counts[counts < 10].index
    df['industry'] = df['industry'].replace(list(rare), 'Other')
else:
    df['industry'] = keyword_industry

df['bill_amount'] = pd.to_numeric(df['invoice_grossRevenue'], errors='coerce').fillna(0.0)

def get_fee_band(bill_amount):
    val = float(bill_amount or 0.0)
    if val < 30000:
        return 'Low-Fee'
    elif val < 100000:
        return 'Standard-Fee'
    return 'Premium-Fee'

df['fee_band'] = df['bill_amount'].apply(get_fee_band)

le_industry = LabelEncoder()
df['industry_encoded'] = le_industry.fit_transform(df['industry'])
le_feeband = LabelEncoder()
df['feeband_encoded'] = le_feeband.fit_transform(df['fee_band'])
le_bd = LabelEncoder()
df['bd_encoded'] = le_bd.fit_transform(df['bdMemberName'].fillna('Unknown'))

## 3. High-Cardinality Frequency Maps & Client Tenure features
Map frequencies for companyName and franchiseeName, and resolve customer tenure ages.

In [4]:
df['alloc_month'] = df['dateOfAllocation'].dt.month
df['alloc_quarter'] = df['dateOfAllocation'].dt.quarter

df['franchiseeName'] = df['franchiseeName'].fillna('Unknown')
df['companyName'] = df['companyName'].fillna('Unknown')
df['franchisee_freq'] = df['franchiseeName'].map(df['franchiseeName'].value_counts())
df['company_freq'] = df['companyName'].map(df['companyName'].value_counts())

df['teamLeaderName'] = df['teamLeaderName'].fillna('Unknown')
le_tl = LabelEncoder()
df['teamlead_encoded'] = le_tl.fit_transform(df['teamLeaderName'])

# Enquiries age anchor based on latest allocation date
as_of = df['dateOfAllocation'].max()
df['enquiry_age_days'] = (as_of - df['dateOfAllocation']).dt.days
df['days_since_invoice'] = (as_of - df['bill_date']).dt.days

# Calculate client relationship tenure
if 'dateClientAcquired' in df.columns:
    df['dateClientAcquired'] = pd.to_datetime(df['dateClientAcquired'], errors='coerce')
    df['client_tenure_days'] = (df['dateOfAllocation'] - df['dateClientAcquired']).dt.days
    df['client_tenure_days'] = df['client_tenure_days'].fillna(df['client_tenure_days'].median())
else:
    df['client_tenure_days'] = 0

print("Advanced feature engineering complete.")

Advanced feature engineering complete.


## 4. Model 1: BD Velocity (Regression)
Optimizes absolute error (MAE) loss using a histogram-based gradient boosting regressor.

In [5]:
mask = (
    df['invoice_billNumber'].notnull() & (df['invoice_billNumber'] != '') &
    df['invoice_billDate'].notnull() &
    df['days_to_close'].notnull() & (df['days_to_close'] >= 0)
)
df_closed = df[mask].copy()
print(f"Closed deals count for velocity training: {len(df_closed)} rows.")

vel_model = None
if len(df_closed) > 0:
    upper = df_closed['days_to_close'].quantile(0.95)
    df_closed['days_to_close_capped'] = df_closed['days_to_close'].clip(upper=upper)
    
    features_vel = ['industry_encoded', 'feeband_encoded', 'bd_encoded',
                     'alloc_month', 'alloc_quarter', 'franchisee_freq',
                     'company_freq', 'teamlead_encoded', 'client_tenure_days']
    X_vel = df_closed[features_vel]
    y_vel = df_closed['days_to_close_capped']

    # Calculate baseline heuristic score
    baseline_pred = df_closed.groupby(['industry_encoded', 'feeband_encoded'])['days_to_close_capped'].transform('median')
    baseline_mae = metrics.mean_absolute_error(y_vel, baseline_pred)
    print(f"Baseline (Industry x FeeBand median) MAE: {baseline_mae:.1f} days")

    X_train, X_test, y_train, y_test = train_test_split(X_vel, y_vel, test_size=0.2, random_state=42)
    vel_model = HistGradientBoostingRegressor(loss='absolute_error', max_iter=300, random_state=42)
    vel_model.fit(X_train, y_train)

    y_pred = vel_model.predict(X_test)
    mae = metrics.mean_absolute_error(y_test, y_pred)
    r2 = metrics.r2_score(y_test, y_pred)
    verdict = "beats" if mae < baseline_mae else "WORSE THAN"
    print(f"BD Velocity Model MAE: {mae:.2f} days (R2: {r2:.3f}) -> {verdict} baseline")

Closed deals count for velocity training: 4919 rows.
Baseline (Industry x FeeBand median) MAE: 87.7 days
BD Velocity Model MAE: 76.49 days (R2: 0.196) -> beats baseline


## 5. Model 2: Leakage Propensity & Payment Collection Risk (Classification)
Random Forest Classifiers using balanced class weights for dataset class-imbalance correction.

In [6]:
df['never_billed'] = (
    (df['enquiryStatus'] == 'closed') &
    (df['invoice_billNumber'].isnull() | (df['invoice_billNumber'] == ''))
).astype(int)

df['billed_payment_pending'] = (
    df['invoice_billNumber'].notnull() & (df['invoice_billNumber'] != '') &
    (df['invoice_amountReceived'] < df['invoice_grossRevenue'] - 1)
).astype(int)

leak_balance = df['never_billed'].value_counts(normalize=True)
pend_balance = df['billed_payment_pending'].value_counts(normalize=True)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### 5A. Leakage Classifier Training

In [7]:
features_leak = ['industry_encoded', 'feeband_encoded', 'bd_encoded',
                  'alloc_month', 'alloc_quarter', 'franchisee_freq',
                  'company_freq', 'teamlead_encoded', 'enquiry_age_days', 'client_tenure_days']
X_leak = df[features_leak]
y_leak = df['never_billed']
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leak, y_leak, test_size=0.2, random_state=42, stratify=y_leak)

leakage_model = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=5, random_state=42,
    class_weight='balanced', n_jobs=-1)
leakage_model.fit(X_train_l, y_train_l)
pred_l = leakage_model.predict(X_test_l)
proba_l = leakage_model.predict_proba(X_test_l)[:, 1]
acc_l = metrics.accuracy_score(y_test_l, pred_l)
prec_l, rec_l, f1_l, _ = metrics.precision_recall_fscore_support(y_test_l, pred_l, average='binary', zero_division=0)
auc_l = metrics.roc_auc_score(y_test_l, proba_l)
cv_auc_l = cross_val_score(leakage_model, X_leak, y_leak, cv=cv, scoring='roc_auc')
verdict_l = "beats" if acc_l > leak_balance.max() else "WORSE THAN"
print(f"Leakage Classifier: accuracy {acc_l*100:.2f}% -> {verdict_l} majority baseline")
print(f"  Precision: {prec_l:.3f} | Recall: {rec_l:.3f} | F1: {f1_l:.3f} | ROC-AUC: {auc_l:.3f}")
print(f"  5-fold CV ROC-AUC: {cv_auc_l.mean():.3f} +/- {cv_auc_l.std():.3f}")

Leakage Classifier: accuracy 81.69% -> beats majority baseline
  Precision: 0.544 | Recall: 0.776 | F1: 0.640 | ROC-AUC: 0.899
  5-fold CV ROC-AUC: 0.894 +/- 0.008


### 5B. Payment Pending Classifier Training

In [8]:
features_pend = ['industry_encoded', 'feeband_encoded', 'bd_encoded',
                  'franchisee_freq', 'company_freq', 'teamlead_encoded',
                  'days_since_invoice', 'client_tenure_days']
X_pend = df[features_pend]
y_pend = df['billed_payment_pending']
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_pend, y_pend, test_size=0.2, random_state=42, stratify=y_pend)

pending_model = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=5, random_state=42,
    class_weight='balanced', n_jobs=-1)
pending_model.fit(X_train_p, y_train_p)
pred_p = pending_model.predict(X_test_p)
proba_p = pending_model.predict_proba(X_test_p)[:, 1]
acc_p = metrics.accuracy_score(y_test_p, pred_p)
prec_p, rec_p, f1_p, _ = metrics.precision_recall_fscore_support(y_test_p, pred_p, average='binary', zero_division=0)
auc_p = metrics.roc_auc_score(y_test_p, proba_p)
cv_auc_p = cross_val_score(pending_model, X_pend, y_pend, cv=cv, scoring='roc_auc')
verdict_p = "beats" if acc_p > pend_balance.max() else "WORSE THAN"
print(f"Payment Pending Classifier: accuracy {acc_p*100:.2f}% -> {verdict_p} majority baseline")
print(f"  Precision: {prec_p:.3f} | Recall: {rec_p:.3f} | F1: {f1_p:.3f} | ROC-AUC: {auc_p:.3f}")
print(f"  5-fold CV ROC-AUC: {cv_auc_p.mean():.3f} +/- {cv_auc_p.std():.3f}")

Payment Pending Classifier: accuracy 84.65% -> beats majority baseline
  Precision: 0.645 | Recall: 0.953 | F1: 0.769 | ROC-AUC: 0.950
  5-fold CV ROC-AUC: 0.950 +/- 0.003


## 6. Serialize and Save Artifacts

In [9]:
os.makedirs('models', exist_ok=True)
if vel_model is not None:
    joblib.dump(vel_model, 'models/velocity_model.joblib')
joblib.dump(leakage_model, 'models/leakage_model.joblib')
joblib.dump(pending_model, 'models/payment_pending_model.joblib')
joblib.dump(le_industry, 'models/le_industry.joblib')
joblib.dump(le_feeband, 'models/le_feeband.joblib')
joblib.dump(le_bd, 'models/le_bd.joblib')
joblib.dump(le_tl, 'models/le_teamlead.joblib')
joblib.dump(df.set_index('franchiseeName')['franchisee_freq'].to_dict(), 'models/franchisee_freq_map.joblib')
joblib.dump(df.set_index('companyName')['company_freq'].to_dict(), 'models/company_freq_map.joblib')

print("All objects successfully serialized and exported to backend/models/")

All objects successfully serialized and exported to backend/models/
